In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import torch 
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score

## 1. Loading the data

In [20]:
# ---------------------------------------------------------
# 1. Load and subsample dataset
# ---------------------------------------------------------

def load_and_subsample(path, n_samples=50000, seed=42):
    df = pd.read_csv(path)
    df = df.sample(n_samples, random_state=seed).reset_index(drop=True)
    return df['puzzle'].values, df['solution'].values

In [21]:
puzzles, solutions = load_and_subsample("data/sudoku.csv", n_samples=5000)
# Example of a puzzle and its solution
puzzles[0], solutions[0]

: 

## 2. Data preprocessing

Here we list the functions that are common for preparing the dataset for all three approaches.

In [8]:
# ---------------------------------------------------------
# 2. Convert 81-char string → 9×9 numpy grid
# ---------------------------------------------------------

def str_to_grid(s):
    arr = np.array([int(c) for c in s]).reshape(9, 9)
    return arr


# ---------------------------------------------------------
# 3. One-hot encode a 9×9 grid (digits 1–9)
#    Empty cells (0) → all zeros
# ---------------------------------------------------------

def one_hot_encode_grid(grid):
    # grid shape: (9, 9)
    one_hot = np.zeros((9, 9, 9), dtype=np.float32)
    for i in range(9):
        for j in range(9):
            d = grid[i, j]
            if d != 0:
                one_hot[i, j, d-1] = 1.0
    return one_hot

In [7]:
# ---------------------------------------------------------
# 4. Train/val/test split (split by puzzles, not by steps)
# ---------------------------------------------------------

def split_puzzles(puzzles, solutions, test_size=0.1, val_size=0.1, seed=42):
    # first split train vs temp
    p_train, p_temp, s_train, s_temp = train_test_split(
        puzzles, solutions, test_size=test_size+val_size, random_state=seed
    )

    # split temp into val and test
    relative_val = val_size / (test_size + val_size)
    p_val, p_test, s_val, s_test = train_test_split(
        p_temp, s_temp, test_size=1-relative_val, random_state=seed
    )

    return (p_train, s_train), (p_val, s_val), (p_test, s_test)

In [9]:
# Split by puzzles
(train_p, train_s), (val_p, val_s), (test_p, test_s) = split_puzzles(puzzles, solutions)

## 3. Sudoku Solving approaches

### 3.1 Step-by-step next-cell prediction

In this approach: 
* Features: current Sudoku grid state (one-hot encoded)
* Target: the correct digit for the next empty cell


In [11]:
# ---------------------------------------------------------
#    Generate step-by-step training samples
#    Strategy: always fill the first empty cell 
# ---------------------------------------------------------

def generate_step_by_step_samples(puzzle, solution):
    puzzle_grid = str_to_grid(puzzle)
    solution_grid = str_to_grid(solution)

    samples_X = []
    samples_y = []

    current = puzzle_grid.copy()

    while True:
        # find first empty cell
        empties = np.argwhere(current == 0)
        if len(empties) == 0:
            break  # puzzle solved

        r, c = empties[0]

        # input = one-hot encoded current grid
        X = one_hot_encode_grid(current)

        # target digit from solution
        y = solution_grid[r, c] - 1  # class 0–8

        samples_X.append(X)
        samples_y.append(y)

        # update grid with the TRUE digit 
        # this is important for training: we want the model to learn from the correct solution path
        current[r, c] = solution_grid[r, c]

    return samples_X, samples_y


# ---------------------------------------------------------
#     Build full dataset of step-by-step samples
# ---------------------------------------------------------

def build_dataset(puzzles, solutions):
    X_all = []
    y_all = []

    for p, s in zip(puzzles, solutions):
        # We generate multiple (X, y) pairs for each puzzle, one for each step of the solution process.
        X_steps, y_steps = generate_step_by_step_samples(p, s)
        X_all.extend(X_steps)
        y_all.extend(y_steps)

    X_all = np.array(X_all)  # shape: (num_samples, 9, 9, 9)
    y_all = np.array(y_all)  # shape: (num_samples,)
    return X_all, y_all


In [ ]:
# Build step-by-step datasets
X_train, y_train = build_dataset(train_p, train_s)
X_val, y_val     = build_dataset(val_p, val_s)


print("Train samples:", X_train.shape)
print("Val samples:", X_val.shape)

Train samples: (1683146, 9, 9, 9)
Val samples: (210365, 9, 9, 9)
Test samples: (210875, 9, 9, 9)


#### 3.1.1 Training the models

#### 3.1.1.1 Random forest and XGBoost

Random forest training

In [ ]:
# Flatten inputs
X_train_flat = X_train.reshape(len(X_train), -1)
X_val_flat   = X_val.reshape(len(X_val), -1)

rf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    max_features="sqrt",
    min_samples_split=5,
    n_jobs=-1,
    random_state=42
)


rf.fit(X_train_flat, y_train)

print("RF val accuracy:", rf.score(X_val_flat, y_val))

MemoryError: could not allocate 150994944 bytes

#### 3.1.2 Testing the models on puzzle test set

In [1]:
def predict_digit(model, grid, flatten=True, is_torch=False, device="cpu"):
    X = one_hot_encode_grid(grid)

    if flatten:
        X = X.reshape(1, -1)
    else:
        X = X.reshape(1, 9, 9, 9)

    if is_torch:
        import torch
        X_t = torch.tensor(X, dtype=torch.float32).to(device)
        with torch.no_grad():
            preds = model(X_t)
            digit = preds.argmax(dim=1).item() + 1
        return digit

    else:
        # sklearn model
        return model.predict(X)[0] + 1

In [3]:
def solve_model(puzzle, solution, model, flatten=True, is_torch=False, device="cpu"):
    grid = str_to_grid(puzzle)
    current = grid.copy()
    solution_grid = str_to_grid(solution)
    correct = 0
    total = 0

    while True:
        empties = np.argwhere(current == 0)
        if len(empties) == 0:
            break

        r, c = empties[0]
        pred = predict_digit(model, current, flatten, is_torch, device)
        total += 1

        if pred == solution_grid[r, c]:
            correct += 1

        current[r, c] = pred

    return current, correct, total

def puzzle_accuracy(pred_grid, solution):
    sol = str_to_grid(solution)
    return (pred_grid == sol).mean()


In [ ]:
# 1. Per-cell accuracy from build_dataset()
X_test, y_test = build_dataset(test_p, test_s)
X_test_flat = X_test.reshape(len(X_test), -1)


y_pred = rf.predict(X_test_flat)
per_cell_acc = accuracy_score(y_test, y_pred)
print("Per-cell accuracy (clean dataset):", per_cell_acc)

# 2. and 3. Per-cell and per-puzzle accuracy during solving
correct_total = 0
total_preds = 0
correct_puzzles = 0

for p, s in zip(test_p, test_s):
    pred_grid, c, t = solve_model(p, s, rf, flatten=True, is_torch=False, device="cpu")
    correct_total += c
    total_preds += t
    if puzzle_accuracy(pred_grid, s) == 1.0:
        correct_puzzles += 1

solver_per_cell_acc = correct_total / total_preds
print("Solver per-cell accuracy:", solver_per_cell_acc)


solver_per_puzzle_acc = correct_puzzles / len(test_p)
print("Solver per-puzzle accuracy:", solver_per_puzzle_acc)



NameError: name 'test_p' is not defined